#### TODO
1. split을 키워드 기반, 문맥 기반 앙상블
2. split을 토큰 기반으로 할 때 recursive-based cursing rag
  - [Advanced RAG and the 3 types of Recursive Retrieval](https://medium.com/enterprise-rag/advanced-rag-and-the-3-types-of-recursive-retrieval-cdd0fa52e1ba)

3. upsatge - document parse 처럼 글씨의 크기를 인식할 수 있는 parser를 찾아보기(pdfminer, pdfflumber(?))

4. fine tunig안할꺼면 gemma - 2- 9b-consafe-lora_v1 활용
  - 전처리(임베딩모델, 추출 방식, chuncking방법, 어떤 vector db에 저장, 유사도 추출방법, 몇개의 chunk를 뽑을지), 리트리버 개선, 모델링하는 방법으로
5. 직접 어떤 문서를 파악하면 좋을지 한번 찾아보기.

python 3.12, torch 2.11


### Setting

In [1]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# We have to check which Torch version for Xformers (2.3 -> 0.0.27)
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton

In [ ]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)
drive.mount('/content/drive')

In [2]:
from huggingface_hub import login
from huggingface_hub import interpreter_login

# Hugging Face API Token 입력 hf_tIHUG~~
# login()
interpreter_login()


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|



### Installation

In [3]:
%%capture
!pip install unsloth
# !pip install unsloth --no-deps

# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git


In [4]:
import torch
torch.cuda.is_available()
torch.__version__

'2.6.0+cu126'

In [5]:
# Install Flash Attention 2 for softcapping support
import torch
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install --no-deps packaging ninja einops "flash-attn>=2.6.3"

  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached flash_attn-2.7.4.post1.tar.gz (6.0 MB)


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\BK\\AppData\\Local\\Temp\\pip-install-9k_wobvy\\flash-attn_dfb6140e28ef47db859b738913aa75bc\\csrc\\composable_kernel\\library\\src\\tensor_operation_instance\\gpu\\batched_gemm_add_relu_gemm_add\\device_batched_gemm_add_relu_gemm_add_xdl_cshuffle_f16_f16_f16_f16_gmk_gnk_gno_gmo_instance.cpp'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [6]:
!pip install datasets
!pip install pymupdf
!pip install -U langchain-community
!pip install faiss-cpu
!pip install -U bitsandbytes

## RAG - Vector DB 생성

### 환경 Import

In [7]:
import os
import glob
import uuid
import torch
import pandas as pd
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import InMemoryStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
from langchain_core.documents import Document

In [8]:
if os.name == "nt":
    src = "C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/"
else:
    src = "/content/drive/MyDrive/Dacon/건설공사 사고 예방 및 대응책 생성/"

### PDF 문서 로드

In [9]:
import re

def remove_kosha_header(text):
    # 정규 표현식을 사용하여 "KOSHA GUIDE C - 숫자 - 숫자" 패턴을 찾아 제거
    pattern = r"KOSHA GUIDE\s*C\s*-\s*\d+\s*-\s*\d+"
    cleaned_text = re.sub(pattern, "", text)
    return cleaned_text

In [10]:
# 경로에 맞게 조정
# pdf_folder = "/content/drive/MyDrive/test/Open/건설안전지침"
pdf_folder = src + "data/건설안전지침/"

# 모든 PDF 파일 가져오기
pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
# 문서 로드
all_docs = []
for pdf_file in pdf_files:
    loader = PyMuPDFLoader(pdf_file)
    docs = loader.load()
    for doc in docs:
        doc.page_content = remove_kosha_header(doc.page_content)
        doc.metadata["source_file"] = os.path.basename(pdf_file)  # 파일 출처 저장
    all_docs.extend(docs)

print(f"총 {len(pdf_files)}개의 PDF에서 {len(all_docs)}개의 문서 로드 완료")

총 104개의 PDF에서 1816개의 문서 로드 완료


### Parent + Child 청크 생성
- parent : 실제 검색에 사용되는 것
- child : 검색된 parent 내부에서 관련 문서를 찾는 기준
- chunk 사이즈가 크면 LLM이 충분한 정보를 가질 수 있으나, 전체적으로 token 소모량 증가, 문서의 수(top_k)를 많이 넣으면 LLM에 따라 contect window를 초과할 수 있어 주의 필요.

#### RecursiveCharacterTextSplitter?
- 긴 문서를 작은 청크로 나누는 도구
- 글자 수를 기준으로 쪼갬
- `chunk_size` : 각 청크의 최대 길이(글자 수)ch
- `chunk_overlap` : 청크 간에 겹치는 부분(글자 수)

In [11]:
parent_text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=50)

# Parent & Child 문서 생성
parent_docs = []
child_docs = []
doc_ids = set()  # 중복 방지를 위해 set 사용

for doc in all_docs:
    doc_id = str(uuid.uuid4())  # 문서별 고유 ID 생성
    doc_ids.add(doc_id)

    # Parent 청크 생성
    parent_chunks = parent_text_splitter.split_documents([doc])
    for chunk in parent_chunks:
        chunk.metadata["doc_id"] = doc_id  # Parent 청크에 문서 ID 추가
    parent_docs.extend(parent_chunks)

    # Child 청크 생성
    child_chunks = child_text_splitter.split_documents([doc])
    for chunk in child_chunks:
        chunk.metadata["doc_id"] = doc_id  # Child 청크에도 동일한 문서 ID 추가 -> Parent chunk와 Child chunk가 같은 문서에서 나온 것임을 구분하기 위해서
    child_docs.extend(child_chunks)

print(f"Parent 청크 개수: {len(parent_docs)}, Child 청크 개수: {len(child_docs)}")


Parent 청크 개수: 2811, Child 청크 개수: 5006


In [12]:
parent_docs

[Document(metadata={'producer': 'ezPDF Builder Supreme', 'creator': '', 'creationdate': '2020-12-23T02:20:00+09:00', 'source': 'C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/data/건설안전지침\\F.C.M 교량공사 안전보건작업 지침.pdf', 'file_path': 'C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/data/건설안전지침\\F.C.M 교량공사 안전보건작업 지침.pdf', 'total_pages': 24, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-12-23T02:20:00+09:00', 'trapped': '', 'modDate': "D:20201223022000+09'00'", 'creationDate': "D:20201223022000+09'00'", 'page': 0, 'source_file': 'F.C.M 교량공사 안전보건작업 지침.pdf', 'doc_id': '2d3b6477-2c59-4502-ae53-3e286db9c4f5'}, page_content='F.C.M 교량공사안전보건작업지침\n2016. 11\n한국산업안전보건공단'),
 Document(metadata={'producer': 'ezPDF Builder Supreme', 'creator': '', 'creationdate': '2020-12-23T02:20:00+09:00', 'source': 'C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/data/건설안전지침\\F.C.M 교량공사 안전보건작업 지침.pdf', 'file_path': 'C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/data/건설안전지침\\F.C.M 교량공사

### 벡터스토어 생성 (Parent + Child 통합)
- Parent와 Child 청크를 하나의 벡터스토어에 저장
- 벡터스토어에서 유사한 청크 찾기 -> 찾은 청크의 doc_id를 청크 메타데이터로 확인 -> doc_id로 inmemeoryStore에서 원본 문서 가져오기

#### 벡터스토어(FAISS)
- FAISS : 실제 벡터를 저장하고 검색하는데 사용되는 라이브러리
- 문서를 잘게 쪼개서 청크로 만든 다음, 각 청크를 벡터로 변환해 저장하는 곳
- 벡터 검색을 할때 사용됨.
- 문서의 의미(의미론적 유사도)를 기반으로 검색할 때 필수

#### InMemoryStore
- 원본 문서 저장
- key : 문서의 고유 id(doc_id)
- valuie : 원본 문서 전체 텍스트


In [13]:
# 어떤 모델로 벡터화할지 설정
embedding_model = "jhgan/ko-sbert-sts"
embedding = HuggingFaceEmbeddings(model_name=embedding_model)

# 기존 벡터스토어 초기화 (중복 방지)
vectorstore_path = src + "test/Open/faiss_vectorstore"
if os.path.exists(vectorstore_path):
    os.system(f"rm -rf {vectorstore_path}")  # 기존 벡터스토어 삭제
    print("⚠ 기존 FAISS 벡터스토어 삭제 완료!")

# 하나의 벡터스토어에 Parent + Child 추가
vectorstore = FAISS.from_documents(parent_docs + child_docs, embedding)

# 원본 문서를 InMemoryStore에 저장
store = InMemoryStore()
store.mset(list(zip(doc_ids, all_docs)))  # 원본 문서 저장 -> key : 문서의 id(doc_id), value : 원본 문서 전체 내용

# MultiVectorRetriever(검색기) 설정 (벡터검색 + 원본 문서 검색)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key="doc_id",
)

print("Parent + Child 통합 벡터스토어 생성 완료!")

C:\Users\BK\AppData\Local\Temp\ipykernel_6360\4119719903.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name=embedding_model)


⚠ 기존 FAISS 벡터스토어 삭제 완료!
Parent + Child 통합 벡터스토어 생성 완료!


### 벡터스토어 저장

In [14]:
vectorstore.save_local(vectorstore_path)
print(f"통합 벡터스토어 저장 완료! 위치: {vectorstore_path}")

통합 벡터스토어 저장 완료! 위치: C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/test/Open/faiss_vectorstore


In [109]:
vectorstore

### 유사도 검색 테스트
- Dense Retriever 활용

In [15]:
import os
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# 저장된 FAISS 벡터스토어 로드
vectorstore_path = src + "test/Open/faiss_vectorstore"

# 임베딩 모델: jhgan/ko-sbert-sts 사용 (저장된 벡터와 동일한 모델 사용 필수!)
embedding_model_name = "jhgan/ko-sbert-sts"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

# FAISS 로드
vector_store = FAISS.load_local(vectorstore_path, embedding, allow_dangerous_deserialization=True)

# 검색기 생성
retriever = vector_store.as_retriever(search_kwargs={"k": 5})  # 가장 유사한 5개 문서 검색

query = """공사종류 대분류 '건축', 중분류 '건축물' 공사 중 공종 대분류 '건축',
중분류 '철근콘크리트공사' 작업에서 사고객체 '건설자재'(중분류: '철근')와 관련된 사고가 발생했습니다.
작업 프로세스는 '설치작업'이며, 사고 원인은 '고소작업 중 추락 위험이 있음에도 불구하고,
안전난간대, 안전고리 착용 등 안전장치가 미흡하였음.'입니다.
재발 방지 대책 및 향후 조치 계획은 무엇인가요?"""

# 검색 실행
retrieved_docs = retriever.invoke(query)

print("\n🔹 [유사도 검색 결과] 🔹")
for i, doc in enumerate(retrieved_docs):
    print(f"\n📌 **유사도 {i+1}위 문서:**")
    print(f"📍 문서 내용:\n{doc.page_content[:500]}")  # 500자까지만 출력 (가독성)
    print(f"📎 출처 정보: {doc.metadata}\n")

print("✅ 유사도 검색 완료!")



🔹 [유사도 검색 결과] 🔹

📌 **유사도 1위 문서:**
📍 문서 내용:
<준수사항>
1. 중량물 취급 작업계획서 : 떨어짐·넘어짐·뒤집힘·깔림·부딪힘·맞음·무너짐·끼임 등의 위험을 예방할 수 있는
안전대책은현장별작업특성에 맞도록작성하여 첨부할 것(관련근거 : 산업안전보건기준에 관한 규칙 
별표4의 11호)
2. 이동식 크레인의 전도 및 침하에 대한 안정성 검토 : 를 참조하여, 양중
작업에 따른 이동식 크레인의 전도 및 침하에 대한 안정성을 검토하고, 그 결과를 첨부할 것
📎 출처 정보: {'producer': 'ezPDF Builder Supreme', 'creator': '', 'creationdate': '2023-08-24T10:05:00+09:00', 'source': 'C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/data/건설안전지침\\건설현장의 중량물 취급 작업계획서(이동식크레인) 작성지침.pdf', 'file_path': 'C:/Users/BK/Desktop/Dacon/건설공사 사고 예방 대응책 생성/data/건설안전지침\\건설현장의 중량물 취급 작업계획서(이동식크레인) 작성지침.pdf', 'total_pages': 16, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-24T10:05:00+09:00', 'trapped': '', 'modDate': "D:20230824100500+09'00'", 'creationDate': "D:20230824100500+09'00'", 'page': 14, 'source_file': '건설현장의 중량물 취급 작업계획서(이동식크레인) 작성지침.pdf', 'doc_id': 'be697f19-a59c-4a93-9c30-833ab31a13ec'}


📌 **유사도 2위 문서:**
📍 문서 내용:
[확인자 :          (서명)         

### cf) 512 청크로만 분할한 코드

In [111]:
import os
import glob
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# PDF 파일들이 저장된 폴더 경로 (예: "건설안전지침")
# pdf_folder = "/content/drive/MyDrive/test/Open/건설안전지침"
pdf_folder = src + "data/건설안전지침/"

# 폴더 내의 모든 PDF 파일 경로 가져오기
pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))

# 모든 PDF 파일에서 문서를 추출하여 리스트에 저장
all_docs = []
for pdf_file in pdf_files:
    loader = PyMuPDFLoader(pdf_file)
    docs = loader.load()  # Document 객체 리스트 반환 (각 문서에 metadata 포함)
    # 각 문서에 파일명 metadata 추가 (추후 출처 확인에 유용)
    for doc in docs:
        doc.metadata["source_file"] = os.path.basename(pdf_file)
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} documents from {len(pdf_files)} PDF files.")

# 문서를 청크로 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
split_docs = text_splitter.split_documents(all_docs)
print(f"Split into {len(split_docs)} text chunks.")

# 임베딩 모델 jhgan/ko-sbert-sts을 사용하여 임베딩 생성
embedding = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-sts")

# FAISS 벡터스토어 생성: 분할된 문서 청크에 임베딩 적용
vectorstore = FAISS.from_documents(split_docs, embedding)
print("Vector store created.")

# (선택 사항) 벡터스토어를 로컬에 저장: 나중에 로드해서 사용할 수 있음
vectorstore.save_local("/content/drive/MyDrive/test/Open/faiss_vectorstore")
print("Vector store saved locally in 'faiss_vectorstore' folder.")


Loaded 1816 documents from 104 PDF files.
Split into 2919 text chunks.
Vector store created.
Vector store saved locally in 'faiss_vectorstore' folder.


## LLM - Dataset 생성

### 01. 데이터 파일 경로 설정

In [112]:
from datasets import Dataset

# 데이터 파일 경로 설정
data_file_path = src + "data/train.csv"

train = pd.read_csv(data_file_path, encoding="utf-8-sig")

### 02. 데이터 전처리
- 공사종류(소분류) 추가
- 층정보 지하, 지상 추가
- 장소, 내외부 추가
- 부위 대분류, 소분류 추가

In [113]:
train["공사종류(대분류)"] = train["공사종류"].str.split(" / ").str[0]
train["공사종류(중분류)"] = train["공사종류"].str.split(" / ").str[1]
train["공종(대분류)"] = train["공종"].str.split(" > ").str[0]
train["공종(중분류)"] = train["공종"].str.split(" > ").str[1]
train["사고객체(대분류)"] = train["사고객체"].str.split(" > ").str[0]
train["사고객체(중분류)"] = train["사고객체"].str.split(" > ").str[1]

In [114]:
# 훈련 데이터 통합 생성
combined_training_data = train.apply(
    lambda row: {
        "question": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'입니다. "
            f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
        ),
        "answer": row["재발방지대책 및 향후조치계획"],
    },
    axis=1,
)

### 03. DataFrame으로 변환

In [115]:
# DataFrame으로 변환
combined_training_data = pd.DataFrame(list(combined_training_data))

combined_training_data.to_csv(
    src + "test/Open/combined_training_data_remove_kosha_header.csv", index=False, encoding="utf-8-sig"
)

### 04. HuggingFace에 업로드

In [116]:
# Pandas DataFrame을 Hugging Face Dataset으로 변환
hf_dataset = Dataset.from_pandas(combined_training_data)

# # 데이터를 train/eval로 분할 (예: 90% train, 10% eval)
# split_dataset = hf_dataset.train_test_split(test_size=0.1, seed=3407)

# # 분할 결과를 DatasetDict 형태로 생성 (평가셋을 "eval" 키로 지정)
# dataset_dict = DatasetDict({
#     "train": split_dataset["train"],
#     "eval": split_dataset["test"]
# })

# 본인의 Dataset 경로 및 공개 여부 설정
dataset_name = "bbboo/ko-construction_safety_prevention_v0.1_remove_kosha_header"
hf_dataset.push_to_hub(dataset_name, private=True)

print(f"Dataset '{dataset_name}' has been uploaded successfully!")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/24 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/316 [00:00<?, ?B/s]

C:\Users\BK\miniconda3\envs\Dacon-RAG-3.11\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BK\.cache\huggingface\hub\datasets--bbboo--ko-construction_safety_prevention_v0.1_remove_kosha_header. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
No files have been modified since last commit. Skipping to pre

Dataset 'bbboo/ko-construction_safety_prevention_v0.1_remove_kosha_header' has been uploaded successfully!


## LLM - Finetunning

### 01. Unsloth
- Hugging Face 기반 모델을 더 빠르고 가볍게 로드할 수 있는 툴킷
- max_seq_length : 한 번에 처리할 최대 토큰 수
- dtype : 데이터 타입, None이면 자동 선택
- load_in_4bit : 4bit 양자화 여부 -> True면 메모리 절약(4bit 모델은 파일 크기가 작기때문)



#### FastLanguageModel.from_pretrained
- Hugging Face의 AutoModelForCasualLM.from_pretrained()랑 비슷하지만 훨씬 빠르고, 메모리를 덜 사용

**기능들**
1. 모델 자동 다운로드 : Hugging Face에서 모델 자동 다운로드
2. 로컬 캐싱 : 한번 받으면 캐싱해서 재다운로드 방지
3. 4bit 로드 : 4bit로 로드
4. 자동 RoPE Scaling : max_seq_length에 맞춰서 자동으로 RoPE(Rotary Position Embedding, 위치 인코딩) 처리
5. 최적 데이터 타입 선택 : dtype를 자동으로 적절히 선택

**Token Option**  
Gated 모델을 사용할 때 필요한 파라미터.  
Gated 모델은 일반적인 모델과 다르게 특별한 액세스 권한이 필요한 모델.  




In [131]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

# 모델 불러오기
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-9b",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

C:\Users\BK\AppData\Local\Temp\ipykernel_19304\3909192214.py:1: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: If you want to finetune Gemma 2, install flash-attn to make it faster!
To install flash-attn, do the below:

pip install --no-deps --upgrade "flash-attn>=2.6.3"
==((====))==  Unsloth 2025.3.9: Fast Gemma2 patching. Transformers: 4.49.0.
   \\   /|    NVIDIA GeForce RTX 3060 Ti. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.13G [00:00<?, ?B/s]

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!  
**LoRa(Low-Rank Adaption)**
- 큰 모델을 효율적으로 학습할 수 있도록 도와주는 기술, 파라미터 수를 줄이면서도 성능을 유지하는데 유용.  
- 기존 fine-tuning 방식에서는 모델의 전체 파라미터를 업데이트하였음. LoRA는 일부 파라미터만을 업데이트하여 모델을 fine-tuning하여 효율적인 학습이 가능하고, 메모리 사용량과 학습 시간을 절감 가능.
-


FastLanguageModel.get_peft_model
- PEFT(Parameter Efficient Fine_Tuning)기법을 활용해 모델을 LoRA 방식으로 수정하는 메소드.
- r : LoRA에서 저차원 랭크를 설정하는 파라미터. 이 값을 높일수록 더 정교한 근사가 가능하지만, 계산량이 많아짐.  
- target_modules : LoRA가 적용될 모델의 모듈  
  - q_proj, k_proj, v_proj, o_proj : 각각 쿼리(query), 키(key), 값(value), 출력(output)을 계산하는 파라미터
  - gate_proj, up_proj, down_proj : LoRa가 적용될 게이트와 투영 연산들을 나타냄
- lora_alpha : LoRA의 스케일링 인자, LoRA로 학습한 근사값을 얼마나 강조할지 조절
- lora_dropout : LoRA가 적용된 파라미터들에 드롭아웃을 적용할 확률
- bias : LoRA 적용시 편향을 적용할지 말지를 결정하는 파라미터
- use_gradient_checkpointing: 기울기 체크포인팅을 사용할지 여부를 결정  
  - 기울기 체크포인팅 : 메모리 사용량을 줄이는 기법, 훈련 중에 기울기를 저장하지 않고 필요할때만 계산하여 메모리 절약
  - unsloth : 최적화된 기울기 체크포인팅을 사용할 때 지정하는 값, 긴 문맥 길이에서 메모리 효율성 향상
- use_rslora : Rank Stabilized LoRA를 사용할지 여부를 설정 하는 파라미터
  - Rank Stabilized LoRA : LoRA의 랭크 안정화를 위해 추가적인 기법 적용
- loftq_config : LoftQ라는 기술을 사용할 떼 필요한 설정값
  - LoftQ : LoRA와 결합하여 효율적으로 모델을 압축하는 기법

In [132]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

NameError: name 'model' is not defined

<a name="Data"></a>
### 02. Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/drive/1XamvWYinY6FOSX9GLvnqSjjsNflxdhNc?usp=sharing).

For text completions like novel writing, try this [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing).

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a construction safety expert. Your task is to analyze construction accident cases and generate a prevention and response plan.

### Input:
{}

### Response:
{}"""


EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    inputs       = examples["question"]
    outputs      = examples["answer"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("SeongeonKim/ko-construction_safety_prevention_v0.1", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

<a name="Train"></a>
### 03. Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        # max_steps=60,
        num_train_epochs=1,  # 추가: 학습 에폭 수 지정
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",  # Use this for WandB etc
    ),
)


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# 학습 시작
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### 04. Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
from unsloth import FastLanguageModel
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "You are a construction safety expert. Your task is to analyze construction accident cases and generate a prevention and response plan.",
        "공사종류 대분류 '건축', 중분류 '건축물' 공사 중 공종 대분류 '건축', 중분류 '해체 및 철거공사' 작업에서 사고객체 '가시설'(중분류: '거푸집')와 관련된 사고가 발생했습니다. 작업 프로세스는 '해체작업'이며, 사고 원인은 '지상1층 해체작업구역에서 벽체 1단 폼(높이: 1200mm)을 해체하던 중 2단에 설치되어 있던 폼이 함께 떨어지며 1차로 재해자의 안전모에 부딪힌 후 2차로 어깨에 타박을 입힘'입니다. 재발 방지 대책 및 향후 조치 계획은 무엇인가요?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

<a name="Save"></a>
### 05. Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# model.save_pretrained("lora_model")
# tokenizer.save_pretrained("lora_model")
# model.push_to_hub("SeongeonKim/gemma-2-9b-ConSafe-LoRA_v1", token = "hf_tIHUG~~") # Online saving
# tokenizer.push_to_hub("SeongeonKim/gemma-2-9b-ConSafe-LoRA_v1", token = "hf_tIHUG~~") # Online saving

# bk token
model.push_to_hub("SeongeonKim/gemma-2-9b-ConSafe-LoRA_v1", token = "hf_bt~~") # Online saving
tokenizer.push_to_hub("SeongeonKim/gemma-2-9b-ConSafe-LoRA_v1", token = "hf_bt~~") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="lora_model",  # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
    [
        alpaca_prompt.format(
            "What is a famous tall tower in Paris?",  # instruction
            "",  # input
            "",  # output - leave this blank for generation!
        )
    ],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
tokenizer.batch_decode(outputs)

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if True:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "SeongeonKim/gemma-2-9b-ConSafe-LoRA_v1", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = True,
    )
    tokenizer = AutoTokenizer.from_pretrained("SeongeonKim/gemma-2-9b-ConSafe-LoRA_v1")

### 06. Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if True:
    model.save_pretrained_merged("model", tokenizer, save_method="merged_16bit")

if True:
    model.push_to_hub_merged(
        "SeongeonKim/gemma-2-9b-ConSafe_v1",
        tokenizer,
        save_method="merged_16bit",
        # private=True  # private 저장 옵션
    )

## RAG + LLM - Inference

### 01. 라이브러리 및 데이터 로드

In [16]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

C:\Users\BK\AppData\Local\Temp\ipykernel_6360\2800618006.py:8: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 02. 벡터 DB 로드 (1024, 512, 256 각각 로드)

In [134]:
vectorstore_paths = {
    "1024": "/content/drive/MyDrive/test/Open/faiss_vectorstore_1024",
    "512": "/content/drive/MyDrive/test/Open/faiss_vectorstore_512",
    "256": "/content/drive/MyDrive/test/Open/faiss_vectorstore_256",
}

# "jhgan/ko-sbert-sts" 임베딩 모델 사용
embedding_model_name = "jhgan/ko-sbert-sts"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

# 벡터스토어 로드 및 Retriever 설정
retrievers = {}
for size, path in vectorstore_paths.items():
    vector_store = FAISS.load_local(path, embedding, allow_dangerous_deserialization=True)
    retrievers[size] = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5})

print("모든 벡터스토어 로드 완료!")

RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss-wheels\faiss-wheels\faiss\faiss\impl\io.cpp:68: Error: 'f' failed: could not open \content\drive\MyDrive\test\Open\faiss_vectorstore_1024\index.faiss for reading: No such file or directory

### 03. Cross-Encoder Reranker 로드

In [ ]:
reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=reranker_model, top_n=3)

print("Cross-Encoder Reranker 로드 완료!")

### 04. Alpaca 스타일 프롬프트 정의

In [ ]:

alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are a construction safety expert specializing in accident prevention and response planning.
Analyze the given accident details and provide a clear and actionable safety measure.
Refer to the provided context to ensure accuracy and compliance with safety regulations.
Focus on key preventive actions and provide direct, practical recommendations.

### Context:
{}

### Question:
{}

### Response:
{}"""

# Response 이후의 텍스트만 추출하는 후처리 함수
def extract_response(text):
    """'Response:' 이후의 텍스트만 추출하는 함수"""
    if "### Response:" in text:
        return text.split("### Response:")[-1].strip()
    return text.strip()

### 05. 테스트 데이터 불러오기

In [135]:
test = pd.read_csv("/content/drive/MyDrive/test/Open/test.csv", encoding="utf-8-sig")

# 테스트 데이터 전처리
test["공사종류(대분류)"] = test["공사종류"].str.split(" / ").str[0]
test["공사종류(중분류)"] = test["공사종류"].str.split(" / ").str[1]
test["공종(대분류)"] = test["공종"].str.split(" > ").str[0]
test["공종(중분류)"] = test["공종"].str.split(" > ").str[1]
test["사고객체(대분류)"] = test["사고객체"].str.split(" > ").str[0]
test["사고객체(중분류)"] = test["사고객체"].str.split(" > ").str[1]

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/test/Open/test.csv'

### 06. 모델 로드 및 토크나이저 설정 (Unsloth 기반)

In [ ]:
model_name = "SeongeonKim/gemma-2-9b-ConSafe-LoRA_v1"
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Fine-tuned LoRA 모델 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_inference(model)  # Unsloth 최적화

print("모델 및 토크나이저 로드 완료!")

### 07. RAG 기반 추론 실행 (Reranker 적용)

In [ ]:
test_results = []

print("테스트 실행 시작... 총 테스트 샘플 수:", len(test))

for idx, row in test.iterrows():
    if (idx + 1) % 50 == 0 or idx == 0:
        print(f"\n[샘플 {idx + 1}/{len(test)}] 진행 중...")

    # 질문 생성
    question = (
        f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
        f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
        f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
        f"작업프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'입니다. "
        f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
    )

    # 다중 벡터스토어 검색 수행
    retrieved_docs = []
    for size, retriever in retrievers.items():
        retrieved_docs.extend(retriever.invoke(question))

    # 중복 제거
    unique_docs = list({doc.page_content: doc for doc in retrieved_docs}.values())

    # Cross-Encoder Reranker 적용 (최종 3개 선택)
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=retriever
    )
    ranked_docs = compression_retriever.invoke(question)

    # 최종 컨텍스트 병합
    context = "\n\n".join([doc.page_content for doc in ranked_docs]) if ranked_docs else "No relevant context found."

    # 질문 + 컨텍스트 조합
    combined_text = alpaca_prompt.format(context, question, "")
    tokenized_text = tokenizer.encode(combined_text, truncation=True, max_length=2048)
    truncated_text = tokenizer.decode(tokenized_text)

    # 저장
    test_results.append(truncated_text)

print("\n테스트 실행 완료! 총 결과 수:", len(test_results))

### 8. 결과 저장 (CSV & 임베딩 추가)

In [ ]:
file_name = "0221_gemma_several_mmr_5_reranked_3.csv"
embedding_sts = SentenceTransformer("jhgan/ko-sbert-sts")
pred_embeddings = embedding_sts.encode(test_results)

submission = pd.read_csv("/content/drive/MyDrive/test/Open/sample_submission.csv", encoding="utf-8-sig")
submission.iloc[:, 1] = test_results
expected_dim = submission.shape[1] - 2
submission.iloc[:, 2:] = pred_embeddings[:, :expected_dim]
submission.to_csv(f"/content/drive/MyDrive/test/Open/{file_name}", index=False, encoding="utf-8-sig")

print(f"최종 결과 저장 완료: {file_name}")
